
# 🧬 Molecular Embedding Generation for ChemTastesDB

This notebook generates multiple types of molecular embeddings for the multi-label taste prediction task.

## Input
- `chemtastesdb_multilabel_clean.csv` - Cleaned dataset with SMILES and multi-label taste annotations

## Output Files (Separate for each embedding type)
| File | Embedding Type | Dimensions | Description |
|------|---------------|------------|-------------|
| `mol2vec.csv` | Mol2Vec | 300 | Molecular fragment embeddings |
| `rdkit_descriptors.csv` | RDKit 2D | ~200 | Physicochemical descriptors |
| `morgan_fps.csv` | Morgan FP | 2048 | Circular fingerprint bits |
| `maccs.csv` | MACCS Keys | 167 | Structural key fingerprints |
| `chemberta.csv` | ChemBERTa | 768 | Transformer-based embeddings |

## Labels (Multi-label)
- Sweet, Bitter, Umami, Sour, Salty

---
Reference - UmamiPred Data generation file

In [1]:
%pip install rdkit pandas numpy scikit-learn transformers torch tqdm joblib gensim mol2vec

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from rdkit import Chem
from mol2vec.features import mol2alt_sentence
from gensim.models import Word2Vec

# ─── Configuration ──────────────────────────────────────────────────────────
INPUT_CSV = "./combined_deduplicated.csv"
MODEL_PATH = "./model_300dim.pkl"
OUTPUT_CSV = "./Embeddings/mol2vec.csv"
LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Undefined']
# ────────────────────────────────────────────────────────────────────────────

def load_and_preprocess_csv(path: str) -> pd.DataFrame:
    """Load combined_deduplicated.csv and convert to multi-label format."""
    df = pd.read_csv(path)
    
    # Use Canonicalized SMILES and rename to canonical SMILES
    df['canonical SMILES'] = df['Canonicalized SMILES']
    
    # Convert single taste column to multi-label columns
    taste_col = 'Canonicalized Taste'
    for label in LABEL_COLS:
        df[label] = (df[taste_col].str.lower() == label.lower()).astype(int)
    
    return df

def smiles_to_sentences(smiles_list, radius: int = 1):
    """Convert SMILES to mol2vec fragment sentences."""
    sentences = []
    invalid_count = 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            sentences.append([])
            invalid_count += 1
        else:
            sentences.append(mol2alt_sentence(mol, radius=radius))
    if invalid_count > 0:
        print(f"       ⚠️ {invalid_count} invalid SMILES found")
    return sentences

def sentences2vec_gensim4(sentences, model: Word2Vec, unseen_vec=None) -> np.ndarray:
    """Custom sentences2vec compatible with Gensim 4.x."""
    vec_dim = model.wv.vector_size
    if unseen_vec is None:
        unseen_vec = np.zeros(vec_dim)
    
    vectors = []
    for sentence in sentences:
        if not sentence:
            vectors.append(unseen_vec)
            continue
        
        word_vecs = []
        for word in sentence:
            if word in model.wv:
                word_vecs.append(model.wv[word])
            else:
                word_vecs.append(unseen_vec)
        
        if word_vecs:
            vectors.append(np.mean(word_vecs, axis=0))
        else:
            vectors.append(unseen_vec)
    
    return np.array(vectors)


if __name__ == "__main__":
    print("=" * 60)
    print("🧬 MOL2VEC EMBEDDING GENERATION")
    print("=" * 60)
    
    print(f"\n[1/5] Loading data from {INPUT_CSV}...")
    df = load_and_preprocess_csv(INPUT_CSV)
    print(f"       Dataset shape: {df.shape}")
    print(f"       Label distribution:")
    for col in LABEL_COLS:
        print(f"         • {col}: {df[col].sum()} positive ({100*df[col].sum()/len(df):.1f}%)")

    print(f"\n[2/5] Loading Mol2Vec model from {MODEL_PATH}...")
    model = Word2Vec.load(MODEL_PATH)
    print(f"       Model vector size: {model.wv.vector_size}")

    print("\n[3/5] Converting SMILES to fragment sentences...")
    sentences = smiles_to_sentences(df['canonical SMILES'], radius=1)

    print("\n[4/5] Computing embeddings...")
    vectors = sentences2vec_gensim4(sentences, model)
    n_dims = vectors.shape[1]
    col_names = [f"mol2vec_{i}" for i in range(n_dims)]
    emb_df = pd.DataFrame(vectors, columns=col_names)

    print(f"\n[5/5] Saving to {OUTPUT_CSV}...")
    out_df = pd.concat([df[LABEL_COLS].reset_index(drop=True), emb_df], axis=1)
    out_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n✅ Done! Output shape: {out_df.shape}")
    print(f"   Columns: {len(LABEL_COLS)} labels + {n_dims} mol2vec features")

🧬 MOL2VEC EMBEDDING GENERATION

[1/5] Loading data from ./combined_deduplicated.csv...
       Dataset shape: (14716, 9)
       Label distribution:
         • Sweet: 9422 positive (64.0%)
         • Bitter: 1515 positive (10.3%)
         • Umami: 192 positive (1.3%)
         • Sour: 1534 positive (10.4%)
         • Undefined: 2053 positive (14.0%)

[2/5] Loading Mol2Vec model from ./model_300dim.pkl...


c:\Users\Parv\OneDrive\Desktop\IP\Multi-Taste Predictor\.venv\Lib\site-packages\gensim\utils.py:1460: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return _pickle.load(f, encoding='latin1')  # needed because loading from S3 doesn't support readline()


       Model vector size: 300

[3/5] Converting SMILES to fragment sentences...


[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerator
[21:22:02] DEPRECATION WARNING: please use MorganGenerat


[4/5] Computing embeddings...

[5/5] Saving to ./Embeddings/mol2vec.csv...


OSError: Cannot save file into a non-existent directory: 'Embeddings'

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# ─── Configuration ──────────────────────────────────────────────────────────
INPUT_CSV = "./combined_deduplicated.csv"
DESCRIPTORS_CSV = "./Embeddings/rdkit_descriptors.csv"
FINGERPRINTS_CSV = "./Embeddings/morgan_fps.csv"
LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Undefined']
# ────────────────────────────────────────────────────────────────────────────

def load_and_preprocess_csv(path: str) -> pd.DataFrame:
    """Load combined_deduplicated.csv and convert to multi-label format."""
    df = pd.read_csv(path)
    
    # Use Canonicalized SMILES and rename to canonical SMILES
    df['canonical SMILES'] = df['Canonicalized SMILES']
    
    # Convert single taste column to multi-label columns
    taste_col = 'Canonicalized Taste'
    for label in LABEL_COLS:
        df[label] = (df[taste_col].str.lower() == label.lower()).astype(int)
    
    return df

def compute_rdkit_descriptors(smiles_list) -> pd.DataFrame:
    """Compute all RDKit 2D molecular descriptors."""
    desc_list = Descriptors.descList
    names = [n for n, _ in desc_list]
    
    records = []
    invalid_count = 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            records.append([np.nan] * len(desc_list))
            invalid_count += 1
        else:
            records.append([func(mol) for _, func in desc_list])
    
    if invalid_count > 0:
        print(f"       ⚠️ {invalid_count} invalid SMILES found")
    
    return pd.DataFrame(records, columns=names)

def compute_morgan_fingerprints(smiles_list, radius: int = 2, n_bits: int = 2048) -> pd.DataFrame:
    """Compute Morgan fingerprints as bit vectors."""
    fps = []
    invalid_count = 0
    
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            arr = np.zeros((n_bits,), dtype=int)
            invalid_count += 1
        else:
            bv = rdMolDescriptors.GetMorganFingerprintAsBitVect(
                mol, radius=radius, nBits=n_bits
            )
            arr = np.array(bv, dtype=int)
        fps.append(arr)
    
    if invalid_count > 0:
        print(f"       ⚠️ {invalid_count} invalid SMILES found")
    
    bitnames = [f"fp_bit_{i}" for i in range(n_bits)]
    return pd.DataFrame(fps, columns=bitnames)


if __name__ == "__main__":
    print("=" * 60)
    print("🧪 RDKIT FEATURES GENERATION")
    print("=" * 60)
    
    print(f"\n[1/4] Loading data from {INPUT_CSV}...")
    df = load_and_preprocess_csv(INPUT_CSV)
    print(f"       Dataset shape: {df.shape}")
    print(f"       Label distribution:")
    for col in LABEL_COLS:
        print(f"         • {col}: {df[col].sum()} positive ({100*df[col].sum()/len(df):.1f}%)")

    print("\n[2/4] Computing RDKit 2D descriptors...")
    desc_df = compute_rdkit_descriptors(df['canonical SMILES'])
    out_desc = pd.concat([df[LABEL_COLS].reset_index(drop=True), desc_df], axis=1)
    out_desc.to_csv(DESCRIPTORS_CSV, index=False)
    print(f"       ✓ Wrote {desc_df.shape[1]} descriptors to {DESCRIPTORS_CSV}")

    print("\n[3/4] Computing Morgan fingerprints (radius=2, bits=2048)...")
    fp_df = compute_morgan_fingerprints(df['canonical SMILES'], radius=2, n_bits=2048)
    out_fp = pd.concat([df[LABEL_COLS].reset_index(drop=True), fp_df], axis=1)
    out_fp.to_csv(FINGERPRINTS_CSV, index=False)
    print(f"       ✓ Wrote {fp_df.shape[1]} fingerprint bits to {FINGERPRINTS_CSV}")

    print("\n[4/4] Summary:")
    print(f"       • Descriptors: {out_desc.shape} → {DESCRIPTORS_CSV}")
    print(f"       • Fingerprints: {out_fp.shape} → {FINGERPRINTS_CSV}")
    print("\n✅ Done!")

🧪 RDKIT FEATURES GENERATION

[1/4] Loading data from ./combined_deduplicated.csv...
       Dataset shape: (14716, 9)
       Label distribution:
         • Sweet: 9422 positive (64.0%)
         • Bitter: 1515 positive (10.3%)
         • Umami: 192 positive (1.3%)
         • Sour: 1534 positive (10.4%)
         • Undefined: 2053 positive (14.0%)

[2/4] Computing RDKit 2D descriptors...


[20:33:56] WARNING: not removing hydrogen atom without neighbors
[20:33:56] WARNING: not removing hydrogen atom without neighbors
[20:35:21] WARNING: not removing hydrogen atom without neighbors
[20:35:21] WARNING: not removing hydrogen atom without neighbors
[20:36:48] WARNING: not removing hydrogen atom without neighbors
[20:36:48] WARNING: not removing hydrogen atom without neighbors
[20:36:53] WARNING: not removing hydrogen atom without neighbors
[20:36:53] WARNING: not removing hydrogen atom without neighbors
[20:37:06] WARNING: not removing hydrogen atom without neighbors
[20:37:06] WARNING: not removing hydrogen atom without neighbors


       ✓ Wrote 217 descriptors to ./Embeddings/rdkit_descriptors.csv

[3/4] Computing Morgan fingerprints (radius=2, bits=2048)...


[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerator
[20:38:35] DEPRECATION WARNING: please use MorganGenerat

       ✓ Wrote 2048 fingerprint bits to ./Embeddings/morgan_fps.csv

[4/4] Summary:
       • Descriptors: (14716, 222) → ./Embeddings/rdkit_descriptors.csv
       • Fingerprints: (14716, 2053) → ./Embeddings/morgan_fps.csv

✅ Done!


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

# ─── Configuration ──────────────────────────────────────────────────────────
INPUT_CSV = "./combined_deduplicated.csv"
OUTPUT_CSV = "./Embeddings/chemberta.csv"
LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Undefined']
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"
BATCH_SIZE = 32
# ────────────────────────────────────────────────────────────────────────────

def load_and_preprocess_csv(path: str) -> pd.DataFrame:
    """Load combined_deduplicated.csv and convert to multi-label format."""
    df = pd.read_csv(path)
    
    # Use Canonicalized SMILES and rename to canonical SMILES
    df['canonical SMILES'] = df['Canonicalized SMILES']
    
    # Convert single taste column to multi-label columns
    taste_col = 'Canonicalized Taste'
    for label in LABEL_COLS:
        df[label] = (df[taste_col].str.lower() == label.lower()).astype(int)
    
    return df

def compute_chemberta_embeddings(smiles_list, model_name=MODEL_NAME, 
                                  batch_size=BATCH_SIZE, device=None):
    """
    Compute ChemBERTa embeddings for a list of SMILES strings.
    Uses [CLS] token embedding as the molecule representation.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"       Using device: {device}")
    print(f"       Loading model: {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    
    all_embeddings = []
    total = len(smiles_list)
    
    with torch.no_grad():
        for i in range(0, total, batch_size):
            batch_smiles = smiles_list[i:i+batch_size]
            
            # Tokenize
            inputs = tokenizer(
                batch_smiles.tolist() if hasattr(batch_smiles, 'tolist') else list(batch_smiles),
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(device)
            
            # Forward pass
            outputs = model(**inputs)
            
            # Use [CLS] token embedding as molecule representation
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_embeddings)
            
            # Progress update
            processed = min(i + batch_size, total)
            if processed % 500 == 0 or processed == total:
                print(f"       Processed {processed}/{total} molecules ({100*processed/total:.1f}%)")
    
    return np.vstack(all_embeddings)


if __name__ == "__main__":
    print("=" * 60)
    print("🤖 CHEMBERTA EMBEDDING GENERATION")
    print("=" * 60)
    
    print(f"\n[1/4] Loading data from {INPUT_CSV}...")
    df = load_and_preprocess_csv(INPUT_CSV)
    print(f"       Dataset shape: {df.shape}")
    print(f"       Label distribution:")
    for col in LABEL_COLS:
        print(f"         • {col}: {df[col].sum()} positive ({100*df[col].sum()/len(df):.1f}%)")
    
    print("\n[2/4] Computing ChemBERTa embeddings...")
    embeddings = compute_chemberta_embeddings(df['canonical SMILES'])
    
    n_dims = embeddings.shape[1]
    col_names = [f"chemberta_{i}" for i in range(n_dims)]
    
    emb_df = pd.DataFrame(embeddings, columns=col_names)
    print(f"       Embedding dimension: {n_dims}")
    
    print(f"\n[3/4] Saving to {OUTPUT_CSV}...")
    out_df = pd.concat([df[LABEL_COLS].reset_index(drop=True), emb_df], axis=1)
    out_df.to_csv(OUTPUT_CSV, index=False)
    
    print(f"\n[4/4] Summary:")
    print(f"       • Output shape: {out_df.shape}")
    print(f"       • Columns: {len(LABEL_COLS)} labels + {n_dims} ChemBERTa features")
    print("\n✅ Done!")

c:\Users\Parv\OneDrive\Desktop\IP\Multi-Taste Predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🤖 CHEMBERTA EMBEDDING GENERATION

[1/4] Loading data from ./combined_deduplicated.csv...
       Dataset shape: (14716, 9)
       Label distribution:
         • Sweet: 9422 positive (64.0%)
         • Bitter: 1515 positive (10.3%)
         • Umami: 192 positive (1.3%)
         • Sour: 1534 positive (10.4%)
         • Undefined: 2053 positive (14.0%)

[2/4] Computing ChemBERTa embeddings...
       Using device: cpu
       Loading model: seyonec/ChemBERTa-zinc-base-v1
       Processed 4000/14716 molecules (27.2%)
       Processed 8000/14716 molecules (54.4%)


In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import MACCSkeys

# ─── Configuration ──────────────────────────────────────────────────────────
INPUT_CSV = "./combined_deduplicated.csv"
OUTPUT_CSV = "./Embeddings/maccs.csv"
LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Undefined']
# ────────────────────────────────────────────────────────────────────────────

def load_and_preprocess_csv(path: str) -> pd.DataFrame:
    """Load combined_deduplicated.csv and convert to multi-label format."""
    df = pd.read_csv(path)
    
    # Use Canonicalized SMILES and rename to canonical SMILES
    df['canonical SMILES'] = df['Canonicalized SMILES']
    
    # Convert single taste column to multi-label columns
    taste_col = 'Canonicalized Taste'
    for label in LABEL_COLS:
        df[label] = (df[taste_col].str.lower() == label.lower()).astype(int)
    
    return df

def compute_maccs_fingerprints(smiles_list) -> pd.DataFrame:
    """
    Compute MACCS Keys fingerprints (166 bits).
    
    MACCS keys are structural keys that encode specific substructural patterns:
    - Atom counts and types
    - Ring structures
    - Functional groups
    - Bond patterns
    """
    fps = []
    invalid_count = 0
    
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            # MACCS keys have 167 bits (0-166), but bit 0 is always 0
            arr = np.zeros((167,), dtype=int)
            invalid_count += 1
        else:
            maccs_fp = MACCSkeys.GenMACCSKeys(mol)
            arr = np.array(maccs_fp, dtype=int)
        fps.append(arr)
    
    if invalid_count > 0:
        print(f"       ⚠️ {invalid_count} invalid SMILES found")
    
    # MACCS keys are 167 bits (indices 0-166)
    bitnames = [f"maccs_{i}" for i in range(167)]
    return pd.DataFrame(fps, columns=bitnames)


if __name__ == "__main__":
    print("=" * 60)
    print("🔑 MACCS KEYS FINGERPRINT GENERATION")
    print("=" * 60)
    
    print(f"\n[1/3] Loading data from {INPUT_CSV}...")
    df = load_and_preprocess_csv(INPUT_CSV)
    print(f"       Dataset shape: {df.shape}")
    print(f"       Label distribution:")
    for col in LABEL_COLS:
        print(f"         • {col}: {df[col].sum()} positive ({100*df[col].sum()/len(df):.1f}%)")

    print("\n[2/3] Computing MACCS Keys fingerprints (167 bits)...")
    maccs_df = compute_maccs_fingerprints(df['canonical SMILES'])
    
    # Calculate bit statistics
    bit_density = maccs_df.mean().mean() * 100
    active_bits = (maccs_df.sum(axis=0) > 0).sum()
    print(f"       Average bit density: {bit_density:.2f}%")
    print(f"       Active bits (at least 1 molecule): {active_bits}/167")

    print(f"\n[3/3] Saving to {OUTPUT_CSV}...")
    out_df = pd.concat([df[LABEL_COLS].reset_index(drop=True), maccs_df], axis=1)
    out_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n✅ Done!")
    print(f"   Output shape: {out_df.shape}")
    print(f"   Columns: {len(LABEL_COLS)} labels + 167 MACCS key bits")

🔑 MACCS KEYS FINGERPRINT GENERATION

[1/3] Loading data from ./filtered_db.csv...
       Dataset shape: (12823, 10)
       Label distribution:
         • Sweet: 6473 positive (50.5%)
         • Bitter: 2210 positive (17.2%)
         • Umami: 220 positive (1.7%)
         • Sour: 1522 positive (11.9%)
         • Undefined: 2398 positive (18.7%)

[2/3] Computing MACCS Keys fingerprints (167 bits)...


[15:53:16] WARNING: not removing hydrogen atom without neighbors
[15:53:19] WARNING: not removing hydrogen atom without neighbors
[15:53:23] WARNING: not removing hydrogen atom without neighbors
[15:53:23] WARNING: not removing hydrogen atom without neighbors


       Average bit density: 21.61%
       Active bits (at least 1 molecule): 161/167

[3/3] Saving to ./Embeddings/maccs.csv...

✅ Done!
   Output shape: (12823, 172)
   Columns: 5 labels + 167 MACCS key bits


---
# 🧹 Embedding Preprocessing

Apply embedding-specific preprocessing and save cleaned versions back to `embeddings/`.

| Embedding | Steps Applied |
|-----------|--------------|
| **RDKit** | Drop inf/NaN cols → median impute → clip outliers (1–99%) → remove zero-variance → remove high-corr (r>0.95) → standardize |
| **Morgan FP** | Remove zero-variance bits → remove rare bits (<0.1%) |
| **MACCS** | Remove zero-variance bits |
| **Mol2Vec** | Drop NaN rows → clip outliers (1–99%) → remove high-corr (r>0.95) → standardize |
| **ChemBERTa** | Drop NaN rows → remove high-corr (r>0.95) → standardize |

In [ ]:
# ─── Shared Preprocessing Utilities ─────────────────────────────────────────
import pandas as pd
import numpy as np
import os

LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Undefined']
EMB_DIR    = "./embeddings"

def split_labels_features(df):
    """Split DataFrame into labels and feature columns."""
    feat_cols = [c for c in df.columns if c not in LABEL_COLS]
    return df[LABEL_COLS], df[feat_cols], feat_cols

def remove_zero_variance(df, feat_cols):
    """Drop columns with zero variance."""
    variances = df[feat_cols].var()
    to_keep = variances[variances > 0].index.tolist()
    dropped = len(feat_cols) - len(to_keep)
    print(f"  Zero-variance: dropped {dropped} cols → {len(to_keep)} remain")
    return to_keep

def remove_rare_bits(df, feat_cols, threshold=0.01):
    """Drop binary columns where positive rate < threshold."""
    freq = df[feat_cols].mean()
    to_keep = freq[freq >= threshold].index.tolist()
    dropped = len(feat_cols) - len(to_keep)
    print(f"  Rare bits (<{threshold*100:.0f}%): dropped {dropped} cols → {len(to_keep)} remain")
    return to_keep

def remove_high_correlation(df, feat_cols, threshold=0.95):
    """Remove one feature from each pair with |correlation| > threshold."""
    corr = df[feat_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = set()
    for col in upper.columns:
        if any(upper[col] > threshold):
            to_drop.add(col)
    to_keep = [c for c in feat_cols if c not in to_drop]
    print(f"  High-corr (|r|>{threshold}): dropped {len(to_drop)} cols → {len(to_keep)} remain")
    return to_keep

def clip_outliers(df, feat_cols, lo_pct=1, hi_pct=99):
    """Clip feature values to [lo_pct, hi_pct] percentile range."""
    df = df.copy()
    for col in feat_cols:
        lo = np.percentile(df[col], lo_pct)
        hi = np.percentile(df[col], hi_pct)
        df[col] = df[col].clip(lo, hi)
    print(f"  Outlier clipping: clipped to [{lo_pct}-{hi_pct}] percentile")
    return df

def standardize(df, feat_cols):
    """Z-score standardization (mean=0, std=1)."""
    df = df.copy()
    for col in feat_cols:
        mu, sigma = df[col].mean(), df[col].std()
        if sigma > 0:
            df[col] = (df[col] - mu) / sigma
        else:
            df[col] = 0.0
    print(f"  Standardized {len(feat_cols)} features")
    return df

def save_processed(labels, features, feat_cols, path):
    """Concat labels + features and save."""
    out = pd.concat([labels.reset_index(drop=True),
                     features[feat_cols].reset_index(drop=True)], axis=1)
    out.to_csv(path, index=False)
    print(f"  💾 Saved → {path}  ({out.shape[0]} rows × {out.shape[1]} cols)")
    return out

print("✅ Preprocessing utilities loaded")

✅ Preprocessing utilities loaded


In [ ]:
# ─── 1. RDKit Descriptors Preprocessing ─────────────────────────────────────
# Steps: drop inf/NaN cols → median impute → clip outliers → remove zero-var
#        → remove high-corr → standardize
print("=" * 60)
print("🧪 RDKit Descriptors Preprocessing")
print("=" * 60)

df = pd.read_csv(os.path.join(EMB_DIR, "rdkit_descriptors.csv"))
labels, features, feat_cols = split_labels_features(df)
print(f"  Original: {len(feat_cols)} features, {len(df)} rows")

# 1a. Drop columns that have any inf values
inf_cols = [c for c in feat_cols if np.isinf(features[c]).any()]
feat_cols = [c for c in feat_cols if c not in inf_cols]
print(f"  Inf columns: dropped {len(inf_cols)} → {len(feat_cols)} remain")

# 1b. Drop columns that are entirely NaN
all_nan = [c for c in feat_cols if features[c].isna().all()]
feat_cols = [c for c in feat_cols if c not in all_nan]
print(f"  All-NaN columns: dropped {len(all_nan)} → {len(feat_cols)} remain")

# 1c. Median imputation for remaining NaN
nan_count = features[feat_cols].isna().sum().sum()
for col in feat_cols:
    if features[col].isna().any():
        features[col] = features[col].fillna(features[col].median())
print(f"  Median imputed {nan_count} NaN values")

# 1d. Clip outliers to 1-99 percentile
features = clip_outliers(features, feat_cols)

# 1e. Remove zero-variance columns
feat_cols = remove_zero_variance(features, feat_cols)

# 1f. Remove highly correlated features (|r| > 0.95)
feat_cols = remove_high_correlation(features, feat_cols, threshold=0.95)

# 1g. Standardize
features = standardize(features, feat_cols)

# Save
rdkit_out = save_processed(labels, features, feat_cols,
                           os.path.join(EMB_DIR, "rdkit_descriptors.csv"))

🧪 RDKit Descriptors Preprocessing
  Original: 217 features, 12823 rows
  Inf columns: dropped 0 → 217 remain
  All-NaN columns: dropped 0 → 217 remain


C:\Users\Parv\AppData\Local\Temp\ipykernel_1724\1923502144.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features[col] = features[col].fillna(features[col].median())
C:\Users\Parv\AppData\Local\Temp\ipykernel_1724\1923502144.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features[col] = features[col].fillna(features[col].median())
C:\Users\Parv\AppData\Local\Temp\ipykernel_1724\1923502144.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try usin

  Median imputed 6348 NaN values
  Outlier clipping: clipped to [1-99] percentile
  Zero-variance: dropped 35 cols → 182 remain
  High-corr (|r|>0.95): dropped 38 cols → 144 remain
  Standardized 144 features
  💾 Saved → ./embeddings\rdkit_descriptors.csv  (12823 rows × 149 cols)


In [ ]:
# ─── 2. Morgan Fingerprints Preprocessing ───────────────────────────────────
# Steps: remove zero-variance bits → remove rare bits (<1% frequency)
print("=" * 60)
print("🔬 Morgan Fingerprints Preprocessing")
print("=" * 60)

df = pd.read_csv(os.path.join(EMB_DIR, "morgan_fps.csv"))
labels, features, feat_cols = split_labels_features(df)
print(f"  Original: {len(feat_cols)} features, {len(df)} rows")

# 2a. Remove zero-variance bits
feat_cols = remove_zero_variance(features, feat_cols)

# 2b. Remove rare bits (< 1% frequency)
feat_cols = remove_rare_bits(features, feat_cols, threshold=0.001)

# Save (no scaling — already binary)
morgan_out = save_processed(labels, features, feat_cols,
                            os.path.join(EMB_DIR, "morgan_fps.csv"))

🔬 Morgan Fingerprints Preprocessing
  Original: 2048 features, 12823 rows
  Zero-variance: dropped 0 cols → 2048 remain
  Rare bits (<0%): dropped 46 cols → 2002 remain
  💾 Saved → ./embeddings\morgan_fps.csv  (12823 rows × 2007 cols)


In [ ]:
# ─── 3. MACCS Keys Preprocessing ────────────────────────────────────────────
# Steps: remove zero-variance bits only (compact & interpretable — no aggressive filtering)
print("=" * 60)
print("🔑 MACCS Keys Preprocessing")
print("=" * 60)

df = pd.read_csv(os.path.join(EMB_DIR, "maccs.csv"))
labels, features, feat_cols = split_labels_features(df)
print(f"  Original: {len(feat_cols)} features, {len(df)} rows")

# 3a. Remove zero-variance bits
feat_cols = remove_zero_variance(features, feat_cols)

# Save (no scaling — already binary)
maccs_out = save_processed(labels, features, feat_cols,
                           os.path.join(EMB_DIR, "maccs.csv"))

🔑 MACCS Keys Preprocessing
  Original: 167 features, 12823 rows
  Zero-variance: dropped 6 cols → 161 remain
  💾 Saved → ./embeddings\maccs.csv  (12823 rows × 166 cols)


In [ ]:

# ─── 4. Mol2Vec Preprocessing ───────────────────────────────────────────────
# Steps: drop NaN rows → clip outliers → remove high-corr → standardize
print("=" * 60)
print("🧬 Mol2Vec Preprocessing")
print("=" * 60)

df = pd.read_csv(os.path.join(EMB_DIR, "mol2vec.csv"))
labels, features, feat_cols = split_labels_features(df)
print(f"  Original: {len(feat_cols)} features, {len(df)} rows")

# # 4a. Drop rows where embedding is all-zero (failed molecules)
# zero_mask = (features[feat_cols] == 0).all(axis=1)
# if zero_mask.sum() > 0:
#     print(f"  Dropped {zero_mask.sum()} all-zero embedding rows")
#     features = features[~zero_mask].reset_index(drop=True)
#     labels = labels[~zero_mask].reset_index(drop=True)

# 4b. Drop rows with any NaN
nan_mask = features[feat_cols].isna().any(axis=1)
if nan_mask.sum() > 0:
    print(f"  Dropped {nan_mask.sum()} rows with NaN")
    features = features[~nan_mask].reset_index(drop=True)
    labels = labels[~nan_mask].reset_index(drop=True)
else:
    print(f"  No NaN rows found")

# 4c. Clip outliers
features = clip_outliers(features, feat_cols)

# 4d. Remove highly correlated features
feat_cols = remove_high_correlation(features, feat_cols, threshold=0.95)

# 4e. Standardize
features = standardize(features, feat_cols)

# Save
mol2vec_out = save_processed(labels, features, feat_cols,
                             os.path.join(EMB_DIR, "mol2vec.csv"))

🧬 Mol2Vec Preprocessing
  Original: 300 features, 14716 rows
  No NaN rows found
  Outlier clipping: clipped to [1-99] percentile
  High-corr (|r|>0.95): dropped 0 cols → 300 remain
  Standardized 300 features
  💾 Saved → ./embeddings\mol2vec.csv  (14716 rows × 305 cols)


In [ ]:
# ─── 5. ChemBERTa Preprocessing ─────────────────────────────────────────────
# Steps: drop NaN rows → remove high-corr → standardize
print("=" * 60)
print("🤖 ChemBERTa Preprocessing")
print("=" * 60)

df = pd.read_csv(os.path.join(EMB_DIR, "chemberta.csv"))
labels, features, feat_cols = split_labels_features(df)
print(f"  Original: {len(feat_cols)} features, {len(df)} rows")

# 5a. Drop rows where embedding is all-zero (failed tokenization)
zero_mask = (features[feat_cols] == 0).all(axis=1)
if zero_mask.sum() > 0:
    print(f"  Dropped {zero_mask.sum()} all-zero embedding rows")
    features = features[~zero_mask].reset_index(drop=True)
    labels = labels[~zero_mask].reset_index(drop=True)

# 5b. Drop rows with any NaN
nan_mask = features[feat_cols].isna().any(axis=1)
if nan_mask.sum() > 0:
    print(f"  Dropped {nan_mask.sum()} rows with NaN")
    features = features[~nan_mask].reset_index(drop=True)
    labels = labels[~nan_mask].reset_index(drop=True)
else:
    print(f"  No NaN rows found")

# 5c. Remove highly correlated features
feat_cols = remove_high_correlation(features, feat_cols, threshold=0.95)

# 5d. Standardize
features = standardize(features, feat_cols)

# Save
chemberta_out = save_processed(labels, features, feat_cols,
                               os.path.join(EMB_DIR, "chemberta.csv"))

🤖 ChemBERTa Preprocessing
  Original: 768 features, 12823 rows
  No NaN rows found
  High-corr (|r|>0.95): dropped 0 cols → 768 remain
  Standardized 768 features
  💾 Saved → ./embeddings\chemberta.csv  (12823 rows × 773 cols)


In [ ]:
# ─── Summary ────────────────────────────────────────────────────────────────
print("=" * 60)
print("📊 PREPROCESSING SUMMARY")
print("=" * 60)

summary_data = []
for name, path in [("RDKit Descriptors", "rdkit_descriptors.csv"),
                    ("Morgan FP", "morgan_fps.csv"),
                    ("MACCS Keys", "maccs.csv"),
                    ("Mol2Vec", "mol2vec.csv"),
                    ("ChemBERTa", "chemberta.csv")]:
    tmp = pd.read_csv(os.path.join(EMB_DIR, path))
    n_features = len([c for c in tmp.columns if c not in LABEL_COLS])
    summary_data.append({"Embedding": name, "Rows": len(tmp), "Features": n_features,
                         "File": path})

summary = pd.DataFrame(summary_data)
print(summary.to_string(index=False))
print("\n✅ All embeddings preprocessed and saved to ./embeddings/")

📊 PREPROCESSING SUMMARY
        Embedding  Rows  Features                  File
RDKit Descriptors 12823       144 rdkit_descriptors.csv
        Morgan FP 12823      2002        morgan_fps.csv
       MACCS Keys 12823       161             maccs.csv
          Mol2Vec 14716       300           mol2vec.csv
        ChemBERTa 12823       768         chemberta.csv

✅ All embeddings preprocessed and saved to ./embeddings/
